# 🚀 XGBoost Serving API (California Housing)

This notebook demonstrates a **Regression** workflow using a larger dataset.

1. **Dataset**: California Housing (20,640 samples, 8 features).
2. **Model**: XGBoost Regressor (predicting continuous house prices).
3. **Serve**: A schema-agnostic FastAPI service.

In [ ]:
# 1. Install Dependencies
%pip install xgboost fastapi uvicorn scikit-learn pandas numpy

print("✅ Dependencies installed. Restart kernel if needed.")

In [ ]:
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print("✅ Libraries imported.")

## 2. Train on California Housing Data
We use `fetch_california_housing`. This dataset is much larger than Iris, so the model learns real patterns rather than memorizing data.

In [ ]:
# 1. Load Data (20,640 samples)
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

# 2. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train (Regression)
# We use XGBRegressor instead of Classifier
model = xgb.XGBRegressor(objective='reg:squarederror')
model.fit(X_train, y_train)

# 4. Evaluate (RMSE)
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"🎯 Model RMSE: {rmse:.4f} (Lower is better)")
print(f"📊 Typical Price Range: 0.15 to 5.0 (Units of $100k)")

In [ ]:
model.save_model("model.json")
print("💾 Model saved to 'model.json'")

## 3. The Maintenance-Free API
We use `%%writefile` to create `app.py`. 

Notice how we didn't have to manually define `MedInc`, `HouseAge`, etc. in the API. 
The API simply accepts a list of floats `[feature_1, feature_2, ...]`.

In [ ]:
%%writefile app.py
import xgboost as xgb
import numpy as np
import os
from fastapi import FastAPI
from pydantic import BaseModel
from typing import List

app = FastAPI(title="Housing Price API")
model = xgb.XGBRegressor()
MODEL_FILE = "model.json"

# Generic Input Schema
class Payload(BaseModel):
    features: List[float]

@app.on_event("startup")
def load_model():
    if os.path.exists(MODEL_FILE):
        model.load_model(MODEL_FILE)
        print(f"✅ Loaded {MODEL_FILE}")
    else:
        print("⚠️ Model file missing.")

@app.post("/predict")
def predict(payload: Payload):
    # Convert generic list -> numpy array
    vector = np.array([payload.features])
    
    # Predict (Returns a float for regression)
    prediction = float(model.predict(vector)[0])
    return {"estimated_value": prediction}


## 4. Launch Server
To start the server, open a **Terminal** in Jupyter and run:
```bash
uvicorn app:app --host 0.0.0.0 --port 8000
```

In [ ]:
!uvicorn app:app --host 0.0.0.0 --port 8000